In [24]:
%run ./config_api_acto

StatementMeta(, 4f796632-d7d2-4b1a-a465-e0fc47a7407d, 26, Finished, Available, Finished)

In [28]:
import jwt
from datetime import datetime, timezone
import requests
import json
import pandas as pd
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    TimestampType
)
import re


decoded = jwt.decode(TOKEN_OSASCO, options={"verify_signature": False})
exp = datetime.fromtimestamp(decoded["exp"], tz=timezone.utc)

print("Token expira em:", exp)
print("Faltam (horas):", (exp - datetime.now(timezone.utc)).total_seconds() / 3600)

url_dados = "https://actogestaoapi-gdhrfgdfc8bbe8hs.brazilsouth-01.azurewebsites.net/api/Tabela/VisualizarDadosIntermediarios"

headers = {
    "Accept": "application/json, text/plain, */*",
    "Authorization": f"Bearer {TOKEN_OSASCO}",
    "App_Id": APP_ID_OSASCO,
    "ApplicationId": APP_ID_OSASCO,
    "Origin": "https://gestaoosascodigital.acto.net.br",
    "Referer": "https://gestaoosascodigital.acto.net.br/",
    "PARAM_LOGIN": "5103",
    "User-Agent": "Mozilla/5.0 ...",
    "Content-Type": "application/json",
}

def buscar_tabela(payload_str: str) -> pd.DataFrame:
    """Recebe o JSON do --data-raw como string e devolve um DataFrame com os dados."""
    config = json.loads(payload_str)

    resp = requests.post(url_dados, headers=headers, json=config)

    data = resp.json()

    lista_final = []
    if "data" in data and isinstance(data["data"], list):
        for item in data["data"]:
            dados_dict = item.get("dados", {})
            if isinstance(dados_dict, dict):
                for key, lista in dados_dict.items():
                    if isinstance(lista, list):
                        lista_final.extend(lista)


    df = pd.DataFrame(lista_final)

    return df


json_config = """
{"nome":"bi_monitora_oz","solicitacoes":[[{"codCatalogo":13366,"codConfigColCatalogo":0,"nomeServico":"Credenciamento para compartilhamento de câmeras particulares ao Programa Monitora OZ - Inscrição","etapasSelecionadas":{"catalogo":[{"codCatalogo":13366,"etapasDados":{"nomeServico":"13366","etapas":[40898]}}]},"filtros":null,"servicos":[{"codConfigCol":null,"col":"seqFluxo","tit":"Nº Solicitação","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":1,"codCliente":0,"codConfigPagina":null,"etapa":null,"codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":null,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"servico","tit":"Serviço","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":2,"codCliente":0,"codConfigPagina":null,"etapa":null,"codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":null,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"statusFluxo","tit":"Status Fluxo","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":3,"codCliente":0,"codConfigPagina":null,"etapa":null,"codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":null,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"dataCriacao","tit":"Data Finalização","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":4,"codCliente":0,"codConfigPagina":null,"etapa":null,"codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":null,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"dataSolicitacao","tit":"Data Criação","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":5,"codCliente":0,"codConfigPagina":null,"etapa":null,"codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":null,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"solicitante","tit":"Solicitante","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":6,"codCliente":0,"codConfigPagina":null,"etapa":null,"codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":null,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_BAIRRO_INSTALACAO","tit":"Bairro:","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":21,"codCliente":0,"codConfigPagina":null,"etapa":"ABERTURA","codFormularioCampo":873621,"linha":null,"largura":null,"codEtapa":40898,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_CEP_INSTALACAO","tit":"CEP:","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":23,"codCliente":0,"codConfigPagina":null,"etapa":"ABERTURA","codFormularioCampo":873616,"linha":null,"largura":null,"codEtapa":40898,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_CNPJ_REPRESENTANTE","tit":"CNPJ","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":27,"codCliente":0,"codConfigPagina":null,"etapa":"ABERTURA","codFormularioCampo":873564,"linha":null,"largura":null,"codEtapa":40898,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_CPF_INTERESSADO_PESQUISA","tit":"CPF:","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":31,"codCliente":0,"codConfigPagina":null,"etapa":"ABERTURA","codFormularioCampo":873643,"linha":null,"largura":null,"codEtapa":40898,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_CNPJ_REPRESENTANTE_PESQUISA","tit":"CNPJ pesquisa","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":65,"codCliente":0,"codConfigPagina":null,"etapa":"ABERTURA","codFormularioCampo":873644,"linha":null,"largura":null,"codEtapa":40898,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_NOME_SOCIAL_REPRESENTANTE","tit":"Nome Fantasia:","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":140,"codCliente":0,"codConfigPagina":null,"etapa":"ABERTURA","codFormularioCampo":873602,"linha":null,"largura":null,"codEtapa":40898,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_NOME_INTERESSADO","tit":"Nome:","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":155,"codCliente":0,"codConfigPagina":null,"etapa":"ABERTURA","codFormularioCampo":873601,"linha":null,"largura":null,"codEtapa":40898,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null}],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null},{"codCatalogo":13254,"codConfigColCatalogo":0,"nomeServico":"Credenciamento para instalação de totem de videomonitoramento em espaço público - Inscrição","etapasSelecionadas":{"catalogo":[{"codCatalogo":13254,"etapasDados":{"nomeServico":"13254","etapas":[40440]}}]},"filtros":null,"servicos":[{"codConfigCol":null,"col":"seqFluxo","tit":"Nº Solicitação","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":7,"codCliente":0,"codConfigPagina":null,"etapa":null,"codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":null,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"servico","tit":"Serviço","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":8,"codCliente":0,"codConfigPagina":null,"etapa":null,"codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":null,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"statusFluxo","tit":"Status Fluxo","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":9,"codCliente":0,"codConfigPagina":null,"etapa":null,"codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":null,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"dataCriacao","tit":"Data Finalização","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":10,"codCliente":0,"codConfigPagina":null,"etapa":null,"codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":null,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"dataSolicitacao","tit":"Data Criação","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":11,"codCliente":0,"codConfigPagina":null,"etapa":null,"codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":null,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"solicitante","tit":"Solicitante","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":12,"codCliente":0,"codConfigPagina":null,"etapa":null,"codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":null,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_BAIRRO_INSTALACAO","tit":"Bairro:","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":40,"codCliente":0,"codConfigPagina":null,"etapa":"ABERTURA","codFormularioCampo":818637,"linha":null,"largura":null,"codEtapa":40440,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_CEP_INSTALACAO","tit":"CEP:","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":52,"codCliente":0,"codConfigPagina":null,"etapa":"ABERTURA","codFormularioCampo":818632,"linha":null,"largura":null,"codEtapa":40440,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"CBO_CNAE_REPRESENTANTE","tit":"CNAE/ Ramo de atividade:","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":55,"codCliente":0,"codConfigPagina":null,"etapa":"ABERTURA","codFormularioCampo":872643,"linha":null,"largura":null,"codEtapa":40440,"json":null,"codCampo":7,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_CNPJ_REPRESENTANTE","tit":"CNPJ","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":58,"codCliente":0,"codConfigPagina":null,"etapa":"ABERTURA","codFormularioCampo":820489,"linha":null,"largura":null,"codEtapa":40440,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_CNPJ_REPRESENTANTE_PESQUISA","tit":"CNPJ pesquisa","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":59,"codCliente":0,"codConfigPagina":null,"etapa":"ABERTURA","codFormularioCampo":871961,"linha":null,"largura":null,"codEtapa":40440,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_CPF_INTERESSADO","tit":"CPF:","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":63,"codCliente":0,"codConfigPagina":null,"etapa":"ABERTURA","codFormularioCampo":818748,"linha":null,"largura":null,"codEtapa":40440,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_NOME_SOCIAL_REPRESENTANTE","tit":"Nome Fantasia:","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":134,"codCliente":0,"codConfigPagina":null,"etapa":"ABERTURA","codFormularioCampo":818751,"linha":null,"largura":null,"codEtapa":40440,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_NOME_INTERESSADO","tit":"Nome:","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":148,"codCliente":0,"codConfigPagina":null,"etapa":"ABERTURA","codFormularioCampo":818750,"linha":null,"largura":null,"codEtapa":40440,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null}],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null}]],"ativo":true,"dataCriacao":"2025-12-05T11:29:14.366Z","dateDataAlteracao":"2025-12-22T13:54:27.597Z","filtros":[],"campos":[],"filtroData":false,"parametrosFiltrosData":[],"id":"6932c20ac0681b6f1af58582","dateDataAlteracaoFormatada":"22/12/2025"}
"""

df_monitora_oz_raw = buscar_tabela(json_config)

df_monitora_oz = df_monitora_oz_raw.copy()

colunas_hash = df_monitora_oz.filter(like="|").columns.tolist()
colunas_sem_hash = [texto.rsplit("|", 1)[0] for texto in colunas_hash]
df_monitora_oz = df_monitora_oz.drop(columns=colunas_sem_hash).copy()

cols_to_drop = [
    "Bairro:|21",
    "Bairro:|40",
    "CEP:|23",
    "CEP:|52",
    "CNPJ pesquisa",
    "CNPJ|27",
    "CNPJ|58",

    "Nome Fantasia:|134",
    "Nome Fantasia:|140",
    "Nome:|148",
    "Nome:|155",
]


df_monitora_oz["bairro"] = (
    df_monitora_oz[["Bairro:|40", "Bairro:|21"]].bfill(axis=1).iloc[:, 0]
)
df_monitora_oz["cep"] = df_monitora_oz[["CEP:|23", "CEP:|52"]].bfill(axis=1).iloc[:, 0]
df_monitora_oz["cnpj"] = df_monitora_oz[["CNPJ|27", "CNPJ|58"]].bfill(axis=1).iloc[:, 0]
df_monitora_oz["nome_fantasia"] = df_monitora_oz[["Nome Fantasia:|134", "Nome Fantasia:|140"]].bfill(axis=1).iloc[:, 0]
df_monitora_oz["nome_interessado"] = df_monitora_oz[["Nome:|148", "Nome:|155"]].bfill(axis=1).iloc[:, 0]
df_monitora_oz['cpf'] = df_monitora_oz[["CPF:|31", "CPF:|63"]].bfill(axis=1).iloc[:, 0]


df_monitora_oz = df_monitora_oz.drop(columns=cols_to_drop).copy()

df_monitora_oz = df_monitora_oz.rename(
    columns={
        "CNAE/ Ramo de atividade:|55": "cnae",
        "Data Criação": "data_criacao",
        "Data Finalização": "data_finalizacao",
        "Nº Solicitação": "os",
        "Serviço": "servico",
        "Solicitante": "solicitante",
        "Status Fluxo": "status",
    }
)

for col in ["data_criacao", "data_finalizacao"]:
    df_monitora_oz[col] = pd.to_datetime(df_monitora_oz[col], format="ISO8601").dt.date

df_monitora_oz = df_monitora_oz.drop(columns=["CPF:|31", "CPF:|63"]).rename(columns={"data_criacao": "data_solicitacao"})


sdf_monitora_oz = spark.createDataFrame(df_monitora_oz)

(
    sdf_monitora_oz
    .write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_monitora_oz")
)

StatementMeta(, 4f796632-d7d2-4b1a-a465-e0fc47a7407d, 30, Finished, Available, Finished)

Token expira em: 2025-12-23 09:37:44+00:00
Faltam (horas): 19.603121906666665
